# Wayble 처리시설 데이터 마이그레이션

## 개요
- **소스**: rewater.wayble.eco REST API
- **목표**: 처리시설 분석결과 → ETA 데이터베이스 마이그레이션
- **기간**: 2026-04-09 ~ 2026-04-15 (저번주)

## 1단계: 마이그레이션 가이드 읽기

In [ ]:
# 웨이블마이그레이션.md 읽기
with open('/Users/azrael/Documents/ETA/Docs/웨이블마이그레이션.md', 'r', encoding='utf-8') as f:
    doc = f.read()

# 문서 출력
print(doc[:2000])
print('\n... (문서 계속)')

## 2단계: 현재 데이터 상태 확인

In [ ]:
import json
import re
from pathlib import Path

# 다운로드된 데이터 확인
txt_file = Path.home() / 'Documents/ETA/Data/facility_data_last_week.txt'
print(f'데이터 파일 존재: {txt_file.exists()}')
print(f'파일 크기: {txt_file.stat().st_size / 1024 / 1024:.2f} MB')

# 처리시설별 데이터 개수
with open(txt_file, 'r', encoding='utf-8') as f:
    content = f.read()

facilities = ['SITE053', 'SITE065', 'SITE066']
for facility in facilities:
    count = content.count(f'"officeCd": "{facility}"')
    print(f'{facility}: {count}개 레코드')

## 3단계: 데이터 매핑 규칙 확인

### 추출 가능한 분석항목
- ✅ **T-N**: tnDensityMg (mg/L) - 계산 결과값
- ✅ **T-P**: tpDensityMg (mg/L) - 계산 결과값
- ✅ **TOC**: tocTcMg, tocIcMg (mg/L) - 계산 결과값
- ✅ **SS**: (ssMgAfter - ssMgBefore) / (ssMl × ssP) × 1000 - 계산 가능
- ✅ **대장균군**: coliA/coliB - 카운트값
- ❌ **BOD**: bodD1, bodD2 - raw 측정값만 존재 (결과값 없음)

## 4단계: Excel 변환 결과 확인

In [ ]:
from openpyxl import load_workbook

excel_file = Path.home() / 'Documents/ETA/Data/facility_data_last_week_v2.xlsx'
wb = load_workbook(excel_file)

print('=== 생성된 시트 ===' )
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    print(f'\n{sheet_name}: {ws.max_row - 1}개 행')
    
    # 처음 3개 행 표시
    for row_num in range(1, min(4, ws.max_row + 1)):
        row_data = []
        for col in range(1, 6):
            val = ws.cell(row_num, col).value
            if val is None:
                row_data.append('')
            else:
                row_data.append(str(val)[:12])
        print(f"  {row_num}: {' | '.join(row_data)}")

## 5단계: 마이그레이션 준비

### 현재 상태
- ✅ API 데이터 다운로드 완료
- ✅ Excel 변환 완료
- ⏳ DB 마이그레이션 준비 중

### 다음 단계
1. 처리시설_결과 테이블 스키마 확인
2. Python 또는 C#로 마이그레이션 스크립트 작성
3. 데이터 유효성 검사
4. 실제 마이그레이션 실행

## 데이터 샘플 분석

In [ ]:
# 중흥 시설 샘플 데이터
from openpyxl import load_workbook
import pandas as pd

wb = load_workbook(excel_file)
ws = wb['중흥']

# 헤더
headers = []
for col in range(1, 12):
    headers.append(ws.cell(1, col).value)

# 데이터 (처음 5행)
data = []
for row_num in range(2, min(7, ws.max_row + 1)):
    row_data = {}
    for col, header in enumerate(headers, 1):
        val = ws.cell(row_num, col).value
        row_data[header] = val if val is not None else ''
    data.append(row_data)

df = pd.DataFrame(data)
print(df.to_string())